# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
# 0. Setup — installs + Hugging Face auth (token stays in Colab Secrets, never in a cell)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn", "pandas"], check=True)

import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")  # Colab Secrets panel (key icon) — set this once
else:
    import os, getpass
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REPO_ID = "FlyRank/internship-warehouse"
MID_PANEL_MONTH = "2026-03"  # a mid-panel month, per the assignment's warning: never the _sample (final month)

print("DuckDB + Hugging Face auth ready. Working month:", MID_PANEL_MONTH)


DuckDB + Hugging Face auth ready. Working month: 2026-03


## 1. Unit of analysis + time window

**One row = one content item's search performance, summed over one calendar month** (`content_hash_id` × `client_hash_id` × month). I'm aggregating `fact_content_daily_performance`'s daily rows up to month-grain, for a single mid-panel month (`month=2026-03`), rather than working at the raw daily grain — that keeps this notebook's slice roughly comparable to my starter-data lane (which used a 90-day rollup), while still being real warehouse data.

**Tables used:** `fact_content_daily_performance` (the daily fact table, filtered to the `month=2026-03` partition) as the primary source, joined to `dim_content` and `dim_clients` where I need content/client-level context (never for per-row metrics).

**Time window:** the single calendar month `2026-03-01` through `2026-03-31` — chosen because it's mid-panel (not the final month), so I'm not accidentally developing on the natural outcome window of any future label, per the assignment's warning about the `_sample` table.

**What I'd predict or rank (label/proxy):** same idea as my lane (w01/w02) — a binary `underperform_flag`: is this page's monthly CTR below the median CTR of other pages at the same position tier, for pages with enough March impressions to judge fairly. This is defined from this month's own observed numbers, not from any FlyRank product decision flag.

**One thing I deliberately exclude:** rows where `ga4_data_available` isn't `TRUE` get excluded from any GA4-derived feature entirely (sessions, engagement) — not zero-filled, not treated as "no engagement." Per the data dictionary, the flag can also be `NULL` (neither `TRUE` nor `FALSE`), so I filter with `IS TRUE` specifically, never `= FALSE` or `NOT ga4_data_available`, to avoid silently mishandling the NULL rows.


In [9]:
# Confirm which tables actually exist in the release before assuming names/paths
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)
top_level = sorted(set(f.split("/")[0] for f in files))
print("Top-level paths in the release:")
for t in top_level:
    print(" -", t)

# Locate the fact table's month=2026-03 partition and the two flat dimension tables
fact_month_files = [f for f in files if "fact_content_daily_performance" in f and f"month={MID_PANEL_MONTH}" in f]
dim_content_files = [f for f in files if "dim_content" in f and "query" not in f]
dim_clients_files = [f for f in files if "dim_clients" in f]

print(f"\nfact_content_daily_performance, month={MID_PANEL_MONTH}: {len(fact_month_files)} file(s)")
print(f"dim_content: {len(dim_content_files)} file(s)")
print(f"dim_clients: {len(dim_clients_files)} file(s)")

FACT_REL = f"read_parquet('hf://datasets/{REPO_ID}/{{path}}')".format(path=f"fact_content_daily_performance/month={MID_PANEL_MONTH}/*.parquet")
DIM_CONTENT_REL = f"read_parquet('hf://datasets/{REPO_ID}/{dim_content_files[0]}')" if dim_content_files else None
DIM_CLIENTS_REL = f"read_parquet('hf://datasets/{REPO_ID}/{dim_clients_files[0]}')" if dim_clients_files else None
print("\nFact relation:", FACT_REL)
print("dim_content relation:", DIM_CONTENT_REL)
print("dim_clients relation:", DIM_CLIENTS_REL)


Top-level paths in the release:
 - .gitattributes
 - README.md
 - dim_clients.parquet
 - dim_content.parquet
 - fact_content_daily_performance
 - fact_content_daily_performance_sample.parquet
 - fact_content_query_90d.parquet

fact_content_daily_performance, month=2026-03: 1 file(s)
dim_content: 1 file(s)
dim_clients: 1 file(s)

Fact relation: read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
dim_content relation: read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')
dim_clients relation: read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `content_hash_id`, `client_hash_id` | Context | Join/group keys only — pseudonyms, never fed to a model |
| `report_date` | Context | Used to build the month window and to group; not a per-row feature at content-month grain |
| `gsc_avg_position` (aggregated to `avg_position_month`) | Feature | Observed search-result position, known as soon as the month's search data lands — before any future decision |
| impressions/clicks columns (aggregated to `impressions_month`, `clicks_month`) | Feature | Observed GSC counts for the month — nothing from later months touched |
| `ctr_month` (derived: `clicks_month / impressions_month`) | Feature, **with a caveat** | Purely arithmetic from two already-safe features — but see the trap in section 3: I must not also use it (or anything derived identically from it) as BOTH a feature and the source of my label in the same model |
| `underperform_flag` (derived: is `ctr_month` below its position tier's median) | **Label / proxy** | The thing I'd rank pages by — never a feature, since it's literally computed from `ctr_month` |
| `ga4_data_available` | Feature (as a flag) + filter | Whether GA4 metrics can be trusted for this row this month; I filter on it with `IS TRUE` AND may also keep the flag itself as a feature ("do we even have engagement data") |
| GA4-derived metrics on rows where `ga4_data_available IS NOT TRUE` | **Excluded** | Per the contract in section 1: not zero-filled, not usable — excluded from any feature entirely, not just filled with 0 |
| Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, etc.) | Excluded | Not shipped in this release at all — nothing to accidentally strip out, by design |


In [10]:
# Discover real column names on both tables so the field classification above matches reality
print("--- fact_content_daily_performance schema (month=2026-03) ---")
con.sql(f"DESCRIBE SELECT * FROM {FACT_REL} LIMIT 0").show(max_rows=100)

if DIM_CLIENTS_REL:
    print("\n--- dim_clients schema ---")
    con.sql(f"DESCRIBE SELECT * FROM {DIM_CLIENTS_REL} LIMIT 0").show(max_rows=100)


--- fact_content_daily_performance schema (month=2026-03) ---
31 rows, 6 columns (report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month)

--- dim_clients schema ---
9 columns (client_hash_id, is_active, has_gsc_access, has_ga4_access, access_profile, client_created_date, client_updated_date, gsc_data_start, ga4_data_start)


## 3. Verify it with queries (grain, counts, missing values, windows)

Three separate checks, each proving one claim from section 1 rather than assuming it.


In [11]:
# Query A — GRAIN: does one row really equal one (date, client, content) triple?
# Zero rows back proves the grain holds; any row back means duplicates exist.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Rows violating the (date, client, content) grain: {len(grain_check)}")
grain_check


Rows violating the (date, client, content) grain: 0


Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []

**Query B — row count and date span:** how many daily rows are actually in this month's partition, and do the dates really span exactly March 2026?


In [12]:
# Query B — COUNTS + DATE SPAN for our slice
counts_check = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {FACT_REL}
""").df()
counts_check


   row_count   min_date   max_date  n_clients  n_content_items
0    9841378 2026-03-01 2026-03-31         55           331437

**Query C — availability:** filtering `ga4_data_available` with `IS TRUE` (never `= FALSE` or `NOT ...`, since the flag can also be `NULL`) — how many rows actually survive?


In [13]:
# Query C — AVAILABILITY, filtered correctly with IS TRUE
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE AND ga4_data_available IS NOT NULL) AS ga4_false_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null_rows
    FROM {FACT_REL}
""").df()
availability_check


   total_rows  ga4_available_rows  ga4_false_rows  ga4_null_rows
0     9841378              413966         6408671        3018741

## Five features, built from March 2026, each “knowable at the decision moment”

Aggregating the daily fact table up to one row per content item for the month, then five features — each with one line on why it was knowable before any future decision point:

1. **`impressions_month`** — knowable because it's a direct sum of search impressions that already happened during March; nothing from April or later is touched.
2. **`clicks_month`** — same reasoning: a sum of clicks that already occurred within the window.
3. **`avg_position_month`** — the month's average GSC position, computed only from rows where a real position was recorded (excluding "no data" rows) — known as soon as March's search data lands.
4. **`days_with_impressions_month`** — count of distinct days in March with at least one impression; an observed frequency signal, not a future one.
5. **`ga4_data_available_month`** — whether this content/client combination had trustworthy GA4 tracking at all during March (`ANY_VALUE` of the flag, since it repeats per content-day); knowing whether engagement data even exists is itself known immediately, before using any GA4 numbers.

`ctr_month` (clicks/impressions) is NOT in this feature list on purpose — it's the quantity my label gets derived from (section 3's trap below), so it stays out of the honest feature set to avoid circularity.


In [14]:
# Build the content-month feature frame from the March partition
# (real column names confirmed via the DESCRIBE in section 2: gsc_impressions / gsc_clicks, not impressions/clicks)
features_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions)                                    AS impressions_month,
        SUM(gsc_clicks)                                         AS clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_month,
        BOOL_OR(ga4_data_available IS TRUE)                     AS ga4_data_available_month
    FROM {FACT_REL}
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 100   -- minimum volume floor so CTR isn't noise on a handful of impressions
"""
feat = con.sql(features_query).df()
feat["ctr_month"] = 100 * feat["clicks_month"] / feat["impressions_month"]
print(f"Feature frame shape: {feat.shape[0]} rows x {feat.shape[1]} columns")
feat.head(8)


Feature frame shape: 101441 rows x 8 columns


            content_hash_id           client_hash_id  impressions_month  \
0  content_d0dff76c889de68f  client_62f4a7e64f5e0096              181.0   
1  content_d49a012dcb924e31  client_62f4a7e64f5e0096              329.0   
2  content_cec711b02f3bbde6  client_62f4a7e64f5e0096              602.0   
3  content_614baf2af4330bd7  client_62f4a7e64f5e0096              772.0   
4  content_755d951187fcd70a  client_62f4a7e64f5e0096             1858.0   
5  content_225dc9235023be5f  client_62f4a7e64f5e0096              488.0   
6  content_cdd114d71966c437  client_62f4a7e64f5e0096             1888.0   
7  content_50b73974582ead90  client_62f4a7e64f5e0096              109.0   

   clicks_month  avg_position_month  days_with_impressions_month  \
0           0.0            5.331238                           29   
1           0.0            5.177774                           31   
2           4.0            4.428747                           29   
3           1.0            4.685335                 

## The trap: deliberately add a label-derived column, watch the score jump, then delete it

First, define the label honestly: `underperform_flag` = 1 when a page's `ctr_month` sits below the median `ctr_month` of other pages at a comparable position (bucketed into simple tiers here, same idea as the starter-data lane). Then build an **honest** quick score using only the five safe features above (never `ctr_month` itself). Then, on purpose, add `ctr_month` back in as a feature — exactly the trap the assignment wants demonstrated — and watch the score jump toward a suspiciously perfect number, because the model is now just reading the label off a feature that IS the label in disguise. Then remove it and keep the honest score.


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

# Position tiers, simple and readable — same spirit as the starter-data position_tier
def position_tier(p):
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    return "deep"

feat = feat.dropna(subset=["avg_position_month"]).copy()
feat["position_tier"] = feat["avg_position_month"].apply(position_tier)
tier_median_ctr = feat.groupby("position_tier")["ctr_month"].transform("median")
feat["underperform_flag"] = (feat["ctr_month"] < tier_median_ctr).astype(int)

honest_features = ["impressions_month", "clicks_month", "avg_position_month",
                    "days_with_impressions_month", "ga4_data_available_month"]
X_honest = feat[honest_features].astype(float)
y = feat["underperform_flag"]

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_auc = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest quick score (ROC AUC, safe features only): {honest_auc:.3f}")

# --- Now spring the trap: add ctr_month itself, the exact quantity the label is thresholded from ---
leaky_features = honest_features + ["ctr_month"]
X_leaky = feat[leaky_features].astype(float)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_train_l, y_train_l)
leaky_auc = roc_auc_score(y_test_l, leaky_model.predict_proba(X_test_l)[:, 1])
print(f"'Leaky' quick score (ctr_month included): {leaky_auc:.3f}  <- jumps toward 1.0, exactly like notebook 02's trend_pct trap")

# --- Delete the leak, keep the honest number ---
print(f"\nKept for real: honest ROC AUC = {honest_auc:.3f} (ctr_month removed from the feature set again)")


Honest quick score (ROC AUC, safe features only): 0.914
'Leaky' quick score (ctr_month included): 0.997  <- jumps toward 1.0, exactly like notebook 02's trend_pct trap

Kept for real: honest ROC AUC = 0.914 (ctr_month removed from the feature set again)


## 4. Data limits

**Named limitation: a single month can't speak to seasonality, and it silently excludes late-onboarding clients.** I picked `month=2026-03` because it's mid-panel and safe from the outcome-window trap, but that also means: (1) whatever patterns show up here are specific to March — I have no evidence they hold in, say, December, and the lane guide's own history notes that different clients' tracking started at very different times (`dim_clients.gsc_data_start`); (2) any client whose GSC tracking hadn't started yet by March 2026 simply doesn't appear in this slice at all — not as zeros, just absent — so `n_clients` from Query B is a lower bound on the full 104-client roster, not the whole roster. A capstone-grade version of this analysis would repeat this contract across several months and explicitly check whether the client set and the underperform rate are stable before trusting any single month's numbers.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.